# s03 — The `mime` Tool

**What this teaches:** how a tool can expose *file-introspection* capabilities the model has no other way to obtain. The `mime` tool reads the first few bytes of a file, identifies the format via [`gabriel-vasile/mimetype`](https://github.com/gabriel-vasile/mimetype) magic-byte tables, and compares the detected type against the filename extension.

**Concept anchor:** the model can read text, but it cannot *see* bytes. A tool fills that gap — it gives the agent a precise sensor for something the LLM is structurally blind to.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [docs/providers.md](../../docs/providers.md)).
- Write access to the notebook's working directory — we drop a temporary `mystery.txt` there.

## 1. Imports

Same imports as [s02_calc](../s02_calc/s02_calc.ipynb); we still need `fstools` for the full tool set that includes `mime`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

Panic on error so the kernel survives the notebook.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Plant a deceptive file

We write a real PNG header (the `\x89PNG\r\n\x1a\n` magic bytes plus a minimal `IHDR` chunk) into a file with a `.txt` extension. A human glancing at the filename would call it plain text. The `mime` tool, by reading the actual bytes, will spot the mismatch.

In [ ]:
const pngMagic = "\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x06\x00\x00\x00\x1f\x15\xc4\x89"
path := "mystery.txt"
must(os.WriteFile(path, []byte(pngMagic), 0o644))
defer os.Remove(path)

## 4. Construct the model and agent

`fstools.New()` returns the full default tool set — `read`, `write`, `grep`, `glob`, `bash`, `revert`, and crucially `mime`. The instruction nudges the model to run the tool and explain the verdict in one sentence.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s03_mime",
    Description: "MIME detection demo.",
    Model:       llm,
    Instruction: "Use the mime tool on the path given, then explain in one sentence " +
        "whether the file extension matches the actual content.",
    Tools: fstools.New(),
})
must(err)

## 5. Run the agent

The model should call `mime` on `mystery.txt`, see `image/png` come back, then narrate the mismatch in plain English.

In [ ]:
r, err := agentkit.Runner("s03", a)
must(err)

prompt := fmt.Sprintf("Inspect %q with the mime tool and tell me whether the extension is honest.", path)
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- A single `mime` tool call returning `image/png` despite the `.txt` suffix — magic-byte detection in action.
- For image MIME types the tool can also surface dimensions and EXIF metadata; try the variations below.
- For a deeper tour of the file-system tools, see [s05_tools](../s05_tools/s05_tools.ipynb).

## Try it yourself

1. Replace `pngMagic` with the first bytes of a real JPEG (`\xff\xd8\xff`) and re-run — the tool should now report `image/jpeg`.
2. Point the agent at a real file on disk and ask whether the extension is misleading.